# Census Tract Request

In [16]:
3 # Import libraries
import pandas as pd
import geopandas as gpd
import requests
import time

In [17]:
# Read census tract file
ct = gpd.read_file(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\NYC_maps\NYC_CensusTracts_2020.json")
ct.head()

,OBJECTID,CTLabel,BoroCode,BoroName,CT2020,BoroCT2020,CDEligibil,NTAName,NTA2020,CDTA2020,CDTANAME,GEOID,PUMA,Shape__Area,Shape__Length,geometry
0,1,1,1,Manhattan,000100,1000100,I,The Battery-Governors Island-Ellis Island-Libe...,MN0191,MN01,MN01 Financial District-Tribeca (CD 1 Equivalent),36061000100,4121,1.843005e+06,10833.043929,"MULTIPOLYGON (((-74.04388 40.6902, -74.04351 4..."
1,2,14.01,1,Manhattan,001401,1001401,I,Lower East Side,MN0302,MN03,MN03 Lower East Side-Chinatown (CD 3 Equivalent),36061001401,4103,1.006117e+06,5075.332000,"POLYGON ((-73.98837 40.71645, -73.98754 40.716..."
2,3,14.02,1,Manhattan,001402,1001402,E,Lower East Side,MN0302,MN03,MN03 Lower East Side-Chinatown (CD 3 Equivalent),36061001402,4103,1.226206e+06,4459.156019,"POLYGON ((-73.98507 40.71909, -73.98423 40.718..."
3,4,18,1,Manhattan,001800,1001800,I,Lower East Side,MN0302,MN03,MN03 Lower East Side-Chinatown (CD 3 Equivalent),36061001800,4103,2.399277e+06,6391.921174,"POLYGON ((-73.98986 40.72053, -73.98972 40.720..."
4,5,22.01,1,Manhattan,002201,1002201,E,Lower East Side,MN0302,MN03,MN03 Lower East Side-Chinatown (CD 3 Equivalent),36061002201,4103,1.740174e+06,5779.062607,"POLYGON ((-73.97875 40.71994, -73.97879 40.719..."


In [18]:
# API  Key 
with open(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\API_Keys\CensusAPIKey.txt", 'r') as file:
   MyKey = file.read().strip()

## DP04

In [19]:
# API Key and URL
api_key = MyKey

# API for ACS 5-year Data Profile Tables
api_url_2024 = 'https://api.census.gov/data/2024/acs/acs5/profile'

# Tables/variables
variables =  ['group(DP04)']  

In [20]:
# Request
   
geo_code = ['005', '047', '061', '081', '085']
api_list = [api_url_2024]
vintages = [2024]

dfs_by_vintage = {}

for url, year in zip(api_list, vintages):
    rows = []
    for code in geo_code:
        params = {
            "get": ",".join(variables),
            "for": "tract:*", 
            "in": ["state:36", f"county:{code}"],
            "key": api_key
        }
        try:
            resp = requests.get(url, params=params, timeout=60)
            resp.raise_for_status()
            json_data = resp.json()

            headers = json_data[0]
            for vals in json_data[1:]:
                rec = dict(zip(headers, vals))
                rec["vintage"] = year
                rec['sample'] = url.split("/acs/")[1].split("/")[0]     # TEST: Including ACS sample type in the output (1 year or 5 year)
                rec['group'] = variables[0]                             # TEST: Include the variable group. WILL BE AN ISSUE FOR IND. VAR??
                rows.append(rec)

        except requests.RequestException as e:
            print(f"[{year}] Error fetching county {code}: {e}")

        time.sleep(0.15)                                            # be nice to the API

    df_year = pd.DataFrame(rows)
    
    # Optional: make numeric columns numeric
    #non_numeric = {"place", "state", "county", "GEO_ID" , "NAME", "vintage", "sample", "group"}
    #num_cols = [c for c in df_year.columns if c not in non_numeric]
    #df_year[num_cols] = df_year[num_cols].apply(pd.to_numeric, errors="coerce")

    dfs_by_vintage[year] = df_year

In [21]:
df_2024 = dfs_by_vintage[2024]
df_2024.head()

,DP04_0001E,DP04_0001EA,DP04_0001M,DP04_0001MA,DP04_0001PE,DP04_0001PEA,DP04_0001PM,DP04_0001PMA,DP04_0002E,DP04_0002EA,...,DP04_0143PM,DP04_0143PMA,GEO_ID,NAME,state,county,tract,vintage,sample,group
0,0,None,14,None,0,None,-888888888,(X),0,None,...,-888888888,(X),1400000US36005000100,Census Tract 1; Bronx County; New York,36,005,000100,2024,acs5,group(DP04)
1,1484,None,198,None,1484,None,-888888888,(X),1416,None,...,-888888888,(X),1400000US36005000200,Census Tract 2; Bronx County; New York,36,005,000200,2024,acs5,group(DP04)
2,2255,None,280,None,2255,None,-888888888,(X),2218,None,...,-888888888,(X),1400000US36005000400,Census Tract 4; Bronx County; New York,36,005,000400,2024,acs5,group(DP04)
3,2142,None,180,None,2142,None,-888888888,(X),2083,None,...,-888888888,(X),1400000US36005001600,Census Tract 16; Bronx County; New York,36,005,001600,2024,acs5,group(DP04)
4,1160,None,116,None,1160,None,-888888888,(X),1149,None,...,-888888888,(X),1400000US36005001901,Census Tract 19.01; Bronx County; New York,36,005,001901,2024,acs5,group(DP04)


In [22]:
df_2024.to_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\5_yr_ACS_2024_CensusTract\5_yr_ACS_2024_CT_NYC_DP04.csv", index = False)